# Llama3.1-8B-Instruct Reinforcement Learning Demo

This notebook demonstrates training on Llama3.1-8B-Instruct model with either GRPO (Group Relative Policy Optimization) or GSPO (Group Sequence Policy Optimization).

This notebook can run on **TPU v6e-8** or **v5p-8**.

## What is GRPO/GSPO?

GRPO/GSPO is an RL algorithm that enhances reasoning abilities of LLMs by:
1. Generating multiple responses for each prompt
2. Evaluating responses using reward models  
3. Calculating relative advantages to update the policy

The difference is in the loss function - either it's optimizing each token (GRPO) or the whole sequence(GSPO).

## Prerequisites

Before running this notebook, make sure your environment is set up for the method you are using. Follow the [Run MaxText Python Notebooks on TPUs](https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html) guide and complete all steps for your chosen method (Google Colab, VS Code, or Local Jupyter Lab) before proceeding.

If you run into issues, refer to the [Common Pitfalls & Debugging](https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html#common-pitfalls-debugging) section of the guide.

In [ ]:
try:
  import google.colab
  print("Running the notebook on Google Colab")
  IN_COLAB = True
except ImportError:
    print("Running the notebook on Visual Studio or JupyterLab")
    IN_COLAB = False

## Installation: MaxText and Post training Dependencies

**Running the notebook on Visual Studio or JupyterLab:** Before proceeding, create a virtual environment and install the required post-training dependencies by following `Option 3: Installing [tpu-post-train]` in the [MaxText installation guide](https://maxtext.readthedocs.io/en/latest/install_maxtext.html#from-source). Once the environment is set up, ensure the notebook is running within it.

In [ ]:
if IN_COLAB:
    # Clone the MaxText repository
    !git clone https://github.com/AI-Hypercomputer/maxtext.git
    %cd maxtext

    # Install uv, a fast Python package installer
    !pip install uv
    
    # Install MaxText and post-training dependencies
    !uv pip install -e .[tpu-post-train] --resolution=lowest
    !install_tpu_post_train_extra_deps

**Session restart Instructions for Colab:**
1.  Navigate to the menu at the top of the screen.
2.  Click on **Runtime**.
3.  Select **Restart session** from the dropdown menu.

You will be asked to confirm the action in a pop-up dialog. Click on **Yes**.

## Environment Setup

In [ ]:
import datetime
import os
import sys
import subprocess
from etils import epath
import jax

from maxtext.trainers.post_train.rl.train_rl import rl_train
from maxtext.utils.model_creation_utils import setup_configs_and_devices
from maxtext.utils.globals import MAXTEXT_REPO_ROOT, MAXTEXT_PKG_DIR

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
os.environ["SKIP_JAX_PRECOMPILE"] = "1"  # Faster startup for vLLM
# Suppress vLLM logging with a severity level below ERROR
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"


print(f"MaxText installation path: {MAXTEXT_PKG_DIR}")

In [ ]:
if not jax.distributed.is_initialized():
  jax.distributed.initialize()
print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

In [ ]:
if IN_COLAB:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    from huggingface_hub import login
    login()

## Model Configurations

In [ ]:
MODEL_NAME = "llama3.1-8b-Instruct"
RUN_NAME = datetime.datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
LOSS_ALGO="grpo" #  or "gspo-token" if you want to use GSPO

CHAT_TEMPLATE_PATH = f"{MAXTEXT_REPO_ROOT}/src/maxtext/examples/chat_templates/gsm8k_rl.json"
if not epath.Path(CHAT_TEMPLATE_PATH).exists():
    raise FileNotFoundError(f"Chat template not found: {CHAT_TEMPLATE_PATH}")

BASE_OUTPUT_DIRECTORY = f"{MAXTEXT_PKG_DIR}/rl_llama3_output"

# set the path to the model checkpoint (including `/0/items`) or leave empty to download from HuggingFace
MODEL_CHECKPOINT_PATH = ""
if not MODEL_CHECKPOINT_PATH:
   MODEL_CHECKPOINT_PATH = f"{BASE_OUTPUT_DIRECTORY}/llama3_checkpoint"
   print("Model checkpoint will be downloaded from HuggingFace at: ",  MODEL_CHECKPOINT_PATH)
   print("Set MODEL_CHECKPOINT_PATH if you do not wish to download the checkpoint.")

## Download Llama3.1-8B Model Checkpoint from Hugging Face

In [ ]:
if not epath.Path(MODEL_CHECKPOINT_PATH).exists():
    # Install torch for the conversion script
    print("Installing torch...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "torch", "--index-url", "https://download.pytorch.org/whl/cpu"
        ],
        check=True
    )

    # Run checkpoint conversion with environment variables
    print("Converting checkpoint from HuggingFace...")
    env = os.environ.copy()
    env["JAX_PLATFORMS"] = "cpu"

    subprocess.run(
        [
            sys.executable,
            "-m", "maxtext.checkpoint_conversion.to_maxtext",
            f"{MAXTEXT_PKG_DIR}/configs/base.yml",
            f"model_name={MODEL_NAME}",
            f"base_output_directory={MODEL_CHECKPOINT_PATH}",
            "use_multimodal=false",
            "scan_layers=true",
            "skip_jax_distributed_system=True",
        ],
        check=True,
        env=env
    )
    
    MODEL_CHECKPOINT_PATH = os.path.join(MODEL_CHECKPOINT_PATH, "0/items")
else:
    print(f"Model checkpoint exists at {MODEL_CHECKPOINT_PATH}")

## MaxText Configurations

In [ ]:
# Configuration for RL training
config_argv = [
    "",
    f"{MAXTEXT_PKG_DIR}/configs/post_train/rl.yml",
    f"model_name={MODEL_NAME}",
    f"run_name={RUN_NAME}",
    f"chat_template_path={CHAT_TEMPLATE_PATH}",
    f"load_parameters_path={MODEL_CHECKPOINT_PATH}",
    f"base_output_directory={BASE_OUTPUT_DIRECTORY}",
    "debug.rl=True",
    f"rl.loss_algo={LOSS_ALGO}",
    "use_pathways=False",
    "log_config=False",
]

print("✓ Configuration initialized successfully")
print(f"📁 Output directory: {BASE_OUTPUT_DIRECTORY}")
print(f"🤖 Model: {MODEL_NAME}")

## RL Training

In [ ]:
import traceback

print("\n" + "=" * 80)
print(f"🚀 Starting {LOSS_ALGO} Training...")
print("=" * 80)
try:
    rl_train(argv=config_argv, kwargs={})
    print("\n" + "=" * 80)
    print("✅ Training Completed Successfully!")
    print("=" * 80)
except Exception:
    print("\n" + "=" * 80)
    print("❌Training Failed!")
    print("=" * 80)
    traceback.print_exc()
    print("\nFor troubleshooting, refer to the Common Pitfalls & Debugging section:")
    print("https://maxtext.readthedocs.io/en/latest/guides/run_python_notebook.html#common-pitfalls-debugging")
    sys.exit(1)

## 📚 Learn More

- **CLI Usage**: https://maxtext.readthedocs.io/en/latest/tutorials/rl.html
- **Configuration**: See `src/maxtext/configs/rl.yml` for all available options
- **Documentation**: Check `src/maxtext/trainers/post_train/rl/train_rl.py` for the `rl_train` function implementation